In [1]:
from core import IronCAD
import inspect
import ctypes
from ctypes import wintypes
import comtypes.gen.ICAPIIRONCADLib as ApiBase
import tempfile
import comtypes.client

Using IRONCAD version: 2026


# Attach to IRONCAD and get baseapp

In [5]:

IRONCAD = IronCAD()
# attach process
IRONCAD.attach()

IZBaseApp = IRONCAD.get_baseapp()
IIronCADApp = IRONCAD.get_ironcadapp()
IIronCADApp.Visible = True
IronCADWindow = IIronCADApp.HWnd

In [6]:
# print methods for base app interface
help(IZBaseApp)

Help on POINTER(IZBaseApp) in module comtypes._post_coinit.unknwn object:

class POINTER(IZBaseApp)(comtypes.gen._0C40AF17_BFB7_4746_B9E6_AE387A3FDDEF_0_27_0.IZBaseApp, POINTER(IDispatch))
 |  Method resolution order:
 |      POINTER(IZBaseApp)
 |      comtypes.gen._0C40AF17_BFB7_4746_B9E6_AE387A3FDDEF_0_27_0.IZBaseApp
 |      POINTER(IDispatch)
 |      comtypes.automation.IDispatch
 |      POINTER(IUnknown)
 |      IUnknown
 |      _compointer_base
 |      ctypes.c_void_p
 |      _ctypes._SimpleCData
 |      _ctypes._CData
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __getattr__(self, name) from comtypes._post_coinit._cominterface_meta_patcher.case_insensitive.<locals>.CaseInsensitive
 |      Implement case insensitive access to methods and properties
 |
 |  __setattr__(self, name, value) from comtypes._post_coinit._cominterface_meta_patcher.case_insensitive.<locals>.CaseInsensitive
 |      Implement case insensitive access to methods and properties
 |
 |  -------------

### Get HWind and send commands

In [ ]:
COMMAND_BLEND = 53134
WM_COMMAND = 0x0111

# Load user32.dll
user32 = ctypes.windll.user32

user32.PostMessageW(IronCADWindow, WM_COMMAND, COMMAND_BLEND, 0)


In [36]:
doc = IZBaseApp.ActiveDoc
# print(doc.Name)
techdraw = doc.QueryInterface(ApiBase.IZDrawingDoc)
# help(techdraw)
sheet = techdraw.ActiveSheet
views = sheet.GetTDViews()
help(views)


KeyError: 13

## create array of catalogs

In [7]:

# get and print all catalogs
CatalogMgr = IZBaseApp.CatalogMgr

cat_iter = 0
catalogs = []
while True:
    try:
        catalogs.append(CatalogMgr.Catalog(cat_iter))
        cat_iter += 1
    except Exception as e:
        break

print(len(catalogs))

3


In [ ]:
# print methods for catalog interface
help(catalogs[0])

### Print All Open Catalogs

In [8]:
for i in catalogs:
    print(i.Name)

RePak-P
Starter
Tank


## Select a specific catalog

In [9]:
SELECTED_CATALOG_NAME = "Tank"
SELECTED_CATALOG = None
for i in catalogs:
    if i.Name == SELECTED_CATALOG_NAME:
        SELECTED_CATALOG = i
        break
if SELECTED_CATALOG is None:
    raise ValueError(f"Catalog '{SELECTED_CATALOG_NAME}' not found.")
else:
    print(f"Selected Catalog: {SELECTED_CATALOG.Name}")

Selected Catalog: Tank


In [10]:
# Activate the selected catalog
SELECTED_CATALOG.Activate()

0

## Create list of catalog entries

In [11]:
# get and print all entries in catalog

entry_iter = 0
entries = []
while True:
    try:
        entries.append(SELECTED_CATALOG.Entry(entry_iter))
        entry_iter += 1
    except Exception as e:
        break

print(len(entries))

6


In [ ]:
# Print methods for catalog entry interface
help(entries[0])

### print all entries in catalog

In [12]:
for entry in entries:
    print(entry.Name)

3. Tank Lid
2. Tank Ring
6. Base Pad
4. Cone Bottom
1. Tank Shell
5. Legs (4)


In [ ]:
temp_dir = tempfile.gettempdir()
entries[0].ExportIconToFile(False, str(temp_dir + "\\icon.png"))

## Select Entry

In [ ]:
SELECTED_ENTRY_NAME = "Leg Assembly"
SELECTED_ENTRY = None
for i in entries:
    if i.Name == SELECTED_ENTRY_NAME:
        SELECTED_ENTRY = i
        break
if SELECTED_ENTRY is None:
    raise ValueError(f"Entry '{SELECTED_ENTRY_NAME}' not found.")
else:
    print(f"Selected Entry: {SELECTED_ENTRY.Name}")

## Drop Entry

In [ ]:
part = SELECTED_ENTRY.InsertElement()

In [ ]:
izdoc = IZBaseApp.ActiveDoc
scenedoc = izdoc.QueryInterface(ApiBase.IZSceneDoc)
scenedoc.Redraw()